In [ ]:
#import sys
#!{sys.executable} -m pip install plotly      #falls man noch was installieren muss, geht das hiermit recht leicht

In [142]:
import requests
import urllib.parse as up
import pandas
from pandas import json_normalize
base = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobs"
query = {
    "angebotsart": "1",
    "wo": "Berlin",
    #"umkreis": "200",
    #"arbeitszeit": "ho;mj",  # Semikolon bleibt bewusst erhalten
    #"page": "5",
    "size": "250",
    #"pav": "false",
}
# safe=';' sorgt dafür, dass das Semikolon nicht encodiert wird, falls die API es so erwartet
url = f"{base}?{up.urlencode(query, safe=';')}"

headers = {
    "X-API-Key": "jobboerse-jobsuche",
    #"Accept": "application/json",
    #"User-Agent": "Mozilla/5.0",
}

r = requests.get(url, headers=headers, timeout=30)
print("URL:", r.url)
print("Status:", r.status_code, r.reason)
print("CT:", r.headers.get("content-type"))
data= r.json()


URL: https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobs?angebotsart=1&wo=Berlin&size=250
Status: 200 OK
CT: application/json;charset=UTF-8


In [143]:
print(data.keys())

dict_keys(['stellenangebote', 'maxErgebnisse', 'page', 'size', 'woOutput', 'facetten'])


In [144]:
facetten=data["facetten"] #alle möglichen Eigenschaften inklusive der dort vorhanden Daten denke ich mal für Angebotsart 1
facetten                  #aus manchen Eigenschaften werde ich aber noch nicht so richtig schlau, da keine Einheiten gegeben sind

{'befristung': {'counts': {'1': 1098, '2': 21729, '3': 16292},
  'maxCount': 39119},
 'verguetung': {'counts': {'jahr': 3104, 'stunde': 10327}, 'maxCount': 39119},
 'externestellenboersen': {'counts': {'false': 33101, 'true': 6018},
  'maxCount': 39119},
 'behinderung': {'counts': {'false': 25797, 'true': 313}, 'maxCount': 39119},
 'pav': {'counts': {'false': 25990, 'true': 13129}, 'maxCount': 39119},
 'berufsfeld': {'counts': {'Altenpflege': 1146,
   'Angehörige der regulären Streitkräfte in sonstigen Rängen': 7,
   'Arzt- und Praxishilfe': 906,
   'Aus- und Trockenbau, Isolierung, Zimmerei, Glaserei': 255,
   'Bau- und Transportgeräteführung': 449,
   'Bauplanung und -überwachung, Architektur': 893,
   'Berg-, Tagebau und Sprengtechnik': 13,
   'Bestattungswesen': 5,
   'Biologie': 37,
   'Bodenverlegung': 121,
   'Buch- und Kunstantiquitäten, Musikfachhandel': 3,
   'Bühnen- und Kostümbildnerei, Requisite': 2,
   'Büro und Sekretariat': 1460,
   'Chemie': 193,
   'Drucktechnik, Buch

In [145]:
#print(data.get("stellenangebote"))

In [146]:
jobs=data["stellenangebote"]
jobs

[{'beruf': 'Fachinformatiker/in - Systemintegration',
  'titel': 'Fachinformatiker Systemintegration (m/w/d)',
  'refnr': '10001-1002668920-S',
  'arbeitsort': {'ort': 'Berlin',
   'strasse': 'null',
   'region': 'Berlin',
   'land': 'Deutschland',
   'koordinaten': {'lat': 52.5112613, 'lon': 13.4255145},
   'entfernung': '0'},
  'arbeitgeber': 'NConsult GmbH',
  'aktuelleVeroeffentlichungsdatum': '2026-02-26',
  'modifikationsTimestamp': '2026-02-26T08:51:46.333',
  'eintrittsdatum': '2026-02-26',
  'kundennummerHash': 'j0aPmLopen8lCLYNYFVQvSNU_Zt6R8pPPsn8NWUkVbA='},
 {'beruf': 'Fachinformatiker/in - Systemintegration',
  'titel': 'Junior IT-Administrator (m/w/d)',
  'refnr': '10001-1002668890-S',
  'arbeitsort': {'ort': 'Berlin',
   'strasse': 'null',
   'region': 'Berlin',
   'land': 'Deutschland',
   'koordinaten': {'lat': 52.5112613, 'lon': 13.4255145},
   'entfernung': '0'},
  'arbeitgeber': 'NConsult GmbH',
  'aktuelleVeroeffentlichungsdatum': '2026-02-26',
  'modifikationsTimes

In [151]:
df = json_normalize(jobs)
#df

In [152]:
df = df[["beruf",	"titel",	"refnr",	"arbeitgeber",	"aktuelleVeroeffentlichungsdatum",	"modifikationsTimestamp",	"eintrittsdatum",	"arbeitsort.plz",	"arbeitsort.ort",	"arbeitsort.strasse",	"arbeitsort.region",	"arbeitsort.land",	"arbeitsort.koordinaten.lat",	"arbeitsort.koordinaten.lon",	"arbeitsort.ortsteil",	"externeUrl"]]
#df

In [149]:
all_jobs = []

for page in range(1, 41):  # immer Seite 1 als Minimum haben - geht aber auch Seite 41 bis 81
    query["page"] = page   #oben habe ich size 250 gesetzt, daher kriegt man 250 x Seitenanzahl an Ergebnissen
    url = f"{base}?{up.urlencode(query, safe=';')}"  #250 x 40 = 10000
    
    r = requests.get(url, headers=headers, timeout=30)
    data = r.json()
    
    all_jobs.extend(data["stellenangebote"])

df = json_normalize(all_jobs) #erzeugt die Spaltenform die man unten sehen kann
df = df[["beruf",	#alle Spalten die man in der Rückgabe haben will - habe zB die Hashs der spezifischen Personen 
         "titel",	#mal rausgenommen, da wir damit denke eh nichts machen werden
         "refnr",	
         "arbeitgeber",	
         "aktuelleVeroeffentlichungsdatum",	
         "modifikationsTimestamp",	
         "eintrittsdatum",	
         "arbeitsort.plz",	
         "arbeitsort.ort",	
         "arbeitsort.strasse",	
         "arbeitsort.region",	
         "arbeitsort.land",	
         "arbeitsort.koordinaten.lat",	
         "arbeitsort.koordinaten.lon",	
         "arbeitsort.ortsteil",	
         "externeUrl"]]
df

,beruf,titel,refnr,arbeitgeber,aktuelleVeroeffentlichungsdatum,modifikationsTimestamp,eintrittsdatum,arbeitsort.plz,arbeitsort.ort,arbeitsort.strasse,arbeitsort.region,arbeitsort.land,arbeitsort.koordinaten.lat,arbeitsort.koordinaten.lon,arbeitsort.ortsteil,externeUrl
0,Fachinformatiker/in - Systemintegration,Fachinformatiker Systemintegration (m/w/d),10001-1002668920-S,NConsult GmbH,2026-02-26,2026-02-26T08:51:46.333,2026-02-26,NaN,Berlin,null,Berlin,Deutschland,52.511261,13.425515,NaN,NaN
1,Fachinformatiker/in - Systemintegration,Junior IT-Administrator (m/w/d),10001-1002668890-S,NConsult GmbH,2026-02-26,2026-02-26T08:49:38.098,2026-02-26,NaN,Berlin,null,Berlin,Deutschland,52.511261,13.425515,NaN,NaN
2,IT-Administrator/in,IT-Administrator (m/w/d),10001-1002668867-S,NConsult GmbH,2026-02-26,2026-02-26T08:47:45.586,2026-02-26,NaN,Berlin,null,Berlin,Deutschland,52.511261,13.425515,NaN,NaN
3,Fachinformatiker/in - Systemintegration,Senior IT-ADMINISTRATOR (m/w/d),10001-1002668843-S,NConsult GmbH,2026-02-26,2026-02-26T08:45:43.656,2026-02-26,NaN,Berlin,null,Berlin,Deutschland,52.511261,13.425515,NaN,NaN
4,Koch/Köchin,Commis de Cuisine (d/w/m) Vollaziet - 12727,10001-1002678734-S,SV (Deutschland) GmbH,2026-02-27,2026-02-27T10:23:46.076,2026-02-27,10243,Berlin,An der Schillingsbrücke 2,Berlin,Deutschland,52.509861,13.429593,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Kommissionierer/in,Kommissionierer (m/w/d),11852-928976524-S,TimePartner Personalmanagement GmbH,2026-02-26,2026-02-26T16:55:26.173,2026-02-26,12685,Berlin,null,Berlin,Deutschland,52.540295,13.560921,NaN,NaN
9996,Erzieher/in,Erzieher für intensivpädagogische Wohngruppe (...,14413-k27218.2999-S,KJHV/KJSH-Stiftung Kinder- u. Jugendhilfe-Verb...,2026-02-26,2026-02-26T10:35:17.065,2026-03-01,12685,Berlin,null,Berlin,Deutschland,52.540295,13.560921,NaN,NaN
9997,Praxisanleiter/in - Pflegeberufe,freigestellte Praxisanleitung,18213-93i6bk85pl-S,Victor's Consulting & Verwaltungs GmbH,2025-11-21,2026-02-23T18:05:10.1,2026-02-23,10249,Berlin,null,Berlin,Deutschland,52.523644,13.445373,NaN,NaN
9998,Wirtschaftsingenieur/in,Bau-/Projektkaufmann/frau - Betriebswirtin (Ho...,10001-1002578011-S,TECHDATA Projektsteuerung und Projekt- managem...,2026-02-10,2026-02-27T16:17:11.336,2026-02-27,10963,Berlin,Obentrautstr. 72,Berlin,Deutschland,52.497443,13.380419,NaN,NaN


In [150]:
df.to_csv("jobs.csv", index=False) #erzeugt eine Datei mit dem Inhalt der in der Anfrage hierdrüber erhaltenen Werte
                                   #das passiert in dem Verzeichnis wo das Jupyter-Notebook geöffnet ist, falls ihr mal nicht wisst wo es gelandet ist